[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/%E7%AC%AC12%E5%9B%9E_%E6%8E%A2%E7%B4%A2%E3%81%A8%E7%A2%BA%E8%A8%BC_%E6%A9%9F%E4%BC%9A%E6%A0%BC%E5%B7%AE%E3%81%A8%E8%8B%A5%E5%B9%B4%E5%A5%B3%E6%80%A7%E3%81%AE%E7%A7%BB%E5%8B%95.ipynb)

> ☝️ Colabで開くには上のバッジをクリック（リンク先 `OWNER/REPO`・ブランチ `main`・パスは配置先に合わせて書き換えてください）。

# 第12回　探索と確証 ― 機会格差と若年女性の移動

いま「地方の若い女性が東京などの都市へ移っていく」ことが話題になっています。
これを、第11回で扱った**就業構造の個票（県別の女性正規雇用率）**と、
**住民基本台帳人口移動報告（県別の若年女性の転入超過）**を重ねて調べてみます。

### この回のねらい
- 第11回で作った「県別パネル」を使い、2つのデータを**県を鍵に結合**する
- 散布図と相関で関係を**探索**する
- そして最大の論点：**出てきた相関は「結論」か「仮説」か** を考える

> 📌 **この回は「反転授業（事前学習＋確認テスト）」形式です。**
> 1. **授業前に各自でこのノートを実行**してください（§2〜§7をすべて、上から順に）。エラーが出たら潰しておく。
> 2. 実行して出てくる次の数値を**確認・メモ**しておく（**確認テストに出ます**）：
>    - 全県の相関 **r**、**東京を除いた** r、**女性の流出率が男性を上回る県数**。
> 3. **確認テスト（Moodle）を締切（授業の開始前）までに提出**してから授業に臨む。
> 4. **授業はこのノートの解説**と、最後に**傾向スコア・マッチングのハンズオン（§8）**を行います。
>
> ▶ のマークは「**このセルを実行する**」合図です。詰まった箇所はメモして授業の質問に使ってください。

> ⚠️ 就業構造側は**擬似ミクロデータ**です。実数の移動データと突き合わせて得た相関は、
> 授業の練習用であり、**実証研究の結論にはしません**。


## 0. 準備（まず下のセルを実行）

この回では次の2ファイルを使います。
- `ippan_2022shugyou_dataset.csv`（令和4年 就業構造基本調査 一般用ミクロデータ）
- `b01103s.xlsx`（住民基本台帳人口移動報告 表11-3、2024年。**第11回 §9 の手順で e-Stat から取得**）

> 📂 **Colab でのファイルの置き方（重要）**
> Colab は、**最初にセルを1つ実行してランタイムに接続してからでないと、ファイルをアップロードできません。**
> 1. **まず下の準備セルを実行**します（ライブラリの読み込み）。これで**ランタイムが接続**されます。
> 2. 接続できたら、画面**左の「ファイル」📁**に上の**2ファイルをアップロード**します（またはドライブをマウントしてパスを指定）。
> 3. アップロードが終わってから、**§2 以降**のセルを実行します。
>
> 💡 `b01103s.xlsx` が未取得なら、第11回 §9 の手順で先に取得してください。


In [ ]:
# ① まずこのセルを実行（→ランタイムに接続）→ 左の「ファイル」📁 にデータをアップロード → ②以降を実行
try:
    import japanize_matplotlib
except ModuleNotFoundError:
    !pip install -q japanize-matplotlib
    import japanize_matplotlib
import pandas as pd, re
import matplotlib.pyplot as plt

SHUGYO = 'ippan_2022shugyou_dataset.csv'
IDOU   = 'b01103s.xlsx'

## 1. 問い（素朴な仮説）

「女性が正規で働きにくい県ほど、若い女性が出ていくのではないか？」
これは自然な仮説です。もし正しければ、**女性正規雇用率が低い県ほど若年女性の流出が大きい**、
つまり右肩下がり（負の相関）になるはずです。データで確かめましょう。


## 2. 移動データから「県別・女20〜24歳の転入超過」を取り出す ▶

e-Statの統計表は多段ヘッダで、そのままでは読めません（第11回の「実データは整形から」の続きです）。
必要なのは1か所、**列『女・20〜24歳』×都道府県の行**だけです。
転入超過がマイナスなら転出超過＝流出です。

> 💡 **列番号を信じる前に、自分の目でヘッダを確認**するのが鉄則（第11回 §8）。
> 下のセルでヘッダ行（行3＝男女、行4＝年齢）を表示し、女×20〜24歳が**列58**であることを確かめます。


In [ ]:
raw = pd.read_excel(IDOU, sheet_name='b01103', header=None, dtype=str)
print('表全体の形:', raw.shape)
# まずヘッダを目視確認（行3=男女, 行4=年齢）
print('列12 →', raw.iloc[3,12], raw.iloc[4,12])
print('列35 →', raw.iloc[3,35], raw.iloc[4,35])
print('列58 →', raw.iloc[3,58], raw.iloc[4,58], '  ← 女×20〜24歳')

In [ ]:
# 7列目までが識別子（表章項目/国籍/年次/地域 等）、8列目以降が 男女×年齢 の値
body = raw.iloc[6:].copy(); body.columns = range(raw.shape[1])

is_pref = lambda c: bool(re.fullmatch(r'(0[1-9]|[1-3]\d|4[0-7])000', str(c)))
mig = body[(body[1]=='60000') & (body[5].apply(is_pref))].copy()  # 60000=移動者(総数)
mig = mig.rename(columns={6:'都道府県'})
mig['女20_24転入超過'] = pd.to_numeric(mig[58])
mig = mig[['都道府県','女20_24転入超過']].reset_index(drop=True)
print('都道府県数:', len(mig))
mig.sort_values('女20_24転入超過').head(6)  # マイナス=流出が大きい県

## 3. 個票から「県別・女性正規雇用率」と「女20〜24歳人口」 ▶

第11回と同じ要領で、就業構造の個票から県別に集計します。
正規雇用率は職員・従業員（正規＋非正規）を母数に、女性だけで計算します。
あわせて、流出を**率**にするための分母（女20〜24歳人口）もWeight合計で作ります。


In [ ]:
s = pd.read_csv(SHUGYO, encoding='cp932', dtype=str)
s['Weight'] = pd.to_numeric(s['Weight'])

emp = s[s['T_WorkRegular'].isin(['正規の職員・従業員','非正規の職員・従業員'])].copy()
emp['正規W'] = (emp['T_WorkRegular']=='正規の職員・従業員') * emp['Weight']
rate = (emp[emp['T_Gender']=='女'].groupby('T_Prefecture')
        .apply(lambda x: x['正規W'].sum()/x['Weight'].sum()*100, include_groups=False)
        .rename('女性正規率'))

popF = (s[(s['T_Gender']=='女') & (s['T_Age']=='20～24歳')]
        .groupby('T_Prefecture')['Weight'].sum().rename('女20_24人口'))
rate.head()

## 4. 県別パネルに結合し、流出率を作る ▶

都道府県を鍵に3つを結合します。
流出率（％）＝ −転入超過 ÷ 女20〜24歳人口 × 100。正の値が大きいほど流出が大きい県です。
（人口で割るのは、東京のように人数が大きい県が自動的に目立つのを防ぐため。第11回の正規化の考え方です。）


In [ ]:
m = (mig.merge(rate, left_on='都道府県', right_index=True)
        .merge(popF, left_on='都道府県', right_index=True))
m['女流出率'] = -m['女20_24転入超過'] / m['女20_24人口'] * 100
print('結合できた県数:', len(m))
m.sort_values('女流出率', ascending=False)[['都道府県','女性正規率','女流出率']].head(8).round(2)

## 5. 散布図で探索する ▶

x＝女性正規雇用率、y＝若年女性の流出率。素朴な仮説が正しければ右肩下がりのはずです。


In [ ]:
r = m['女性正規率'].corr(m['女流出率'])
fig, ax = plt.subplots(figsize=(7,6))
ax.scatter(m['女性正規率'], m['女流出率'], alpha=0.7)
for _,row in m.iterrows():
    if abs(row['女流出率'])>4 or row['都道府県'] in ['東京都']:
        ax.annotate(row['都道府県'], (row['女性正規率'], row['女流出率']), fontsize=8)
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('女性正規雇用率 (%)'); ax.set_ylabel('若年女性(20-24)の流出率 (%)')
ax.set_title(f'女性正規率 と 若年女性の流出率（相関 r = {r:.3f}）')
plt.show()
print('相関係数 r =', round(r,3))

## 6. 読み取り ― これは「結論」か「仮説」か

予想は右肩下がり（負の相関）でした。ところが実際は **r ≈ 0.24 と弱く、向きはむしろ正**。
「正規で働きにくいから出ていく」という素朴な物語は、**この県単位データでは支持されません**。

ここで結論を急がず、まず**外れ値を疑います**。東京は飛び抜けた点（最高の正規率かつ最大の流入）です。
1つの点が相関をどれだけ動かすか見てみましょう。


In [ ]:
r_all = m['女性正規率'].corr(m['女流出率'])
m2 = m[m['都道府県'] != '東京都']
r_wo = m2['女性正規率'].corr(m2['女流出率'])
print(f'全県        r = {r_all:.3f}')
print(f'東京を除く  r = {r_wo:.3f}')

東京を外すと r が 0.24 → 0.49 へ大きく変わります。**たった1点で相関は動く**。
散布図と外れ値を見ずに相関係数だけ報告してはいけない、という教訓です。

そして符号は依然として正（むしろ強まる）。「機会が乏しいから流出」とは逆です。なぜでしょう。
考えられる説明は複数あり、**どれもこの相関だけでは選べません**（すべて仮説です）：
- 流出の主因は就職より**進学**かもしれない（大学の多い都市へ）。
- 全年齢の正規率は、**若い女性が感じる機会**を捉えていないかもしれない。
- 東京は正規率も高く人も集まる＝**「都市度」が、正規率と流入の両方を同時に押し上げている**かもしれない（下の「交絡」）。
- そもそも県（集団）の平均で見た関係を、そのまま個人にあてはめてはいけない（＝県の相関で個人を語る誤り）。

> 📖 **用語：交絡（こうらく）とは** ―― 教科書には出てこない言葉なので説明します。
> x（ここでは女性正規率）と y（流出率）の**両方に影響する「第3の要因」**が背後にあって、そのせいで x と y が一緒に動いて見えることを **交絡** といいます。
>
> 例：**都市度**（大学や仕事の多さ）が高い県は、女性正規率も高くなりやすく、同時に若い女性も集まりやすい。
> すると「正規率が高い県ほど流入も多い」という関係が出ますが、それは**正規率が流入を生んだのではなく、都市度が両方を動かしているだけ**かもしれません。このとき都市度が**交絡要因**です。
> 交絡があると、相関から「x→y」の因果は読み取れません。確かめるには、**都市度などの条件をそろえて比べる**（交絡を取り除く）工夫が要ります ―― これが第8回でやった**傾向スコア**の発想です。

「効いていそう」を見つけるのが**探索**、それを確かめるのが**確証**。
確証には、個人単位のデータ・進学や産業構成などの**交絡（上の「第3の要因」）をそろえる調整**・時間方向の情報が要ります
（第8回の傾向スコアの発想）。今日できたのは探索＝**仮説生成**までです。


### （発展・教科書外・任意）この相関は「偶然」で説明できる？ ― 統計的検定のごく短い話

教科書では扱っていませんが、相関を見ると必ず出てくる話題に**統計的検定**があります。ここだけ軽く触れます（飛ばして先に進んでも構いません）。

検定が答えるのは、たった**1つの狭い問い**です：

> 「本当は無関係（相関＝0）だとしても、**たまたまのばらつきだけで、これくらいの相関が出てしまう確率**はどれくらいか？」

この確率を **p値** と呼びます。

**素朴に言うと、p＝0.05 は「20回に1回（5%）」という意味です。**
「本当は無関係でも、偶然のばらつきだけで、これくらいの相関が **20回やれば1回くらいは** 出てしまう」ということ。
- よく起こる（ありふれている）なら → 「偶然で説明できる」＝**有意でない**。
- めったに起きないなら → 「偶然では説明しにくい」＝**有意**。

だから p が小さいほど「偶然では苦しい」。慣習では **p<0.05（20回に1回より珍しい）** を「有意」の目安にします。
（例：p＝0.0005 は「**約2000回に1回**」＝めったに起きない。p＝0.11 は「**約9回に1回**」＝わりとよく起こる。）

> 📖 **2つの「相関」の名前について**（教科書では名前までは出てきません）
> - **ピアソンの相関係数**：教科書で単に「相関係数」と呼び、§5で計算したもの（r=0.236）の正式名称です。値の大きさをそのまま使うので、**東京のような極端な点（外れ値）に強く引っぱられます**。
> - **スピアマンの相関係数**（教科書外）：値の大きさではなく**順位（小さい方から何番目か）**だけを使う相関です。順位に直すと極端な値も「1位・2位…」になるだけなので、**外れ値の影響を受けにくい**のが特徴。下では「参考」として並べます。

全県と「東京を除く」で、相関係数（ピアソン）と p値を出してみましょう。


In [ ]:
from scipy import stats   # Colab には最初から入っています

r_all, p_all = stats.pearsonr(m['女性正規率'], m['女流出率'])
m2 = m[m['都道府県'] != '東京都']
r_wo,  p_wo  = stats.pearsonr(m2['女性正規率'], m2['女流出率'])
print(f'全県      : ピアソン r={r_all:.3f},  p={p_all:.4f}')
print(f'東京を除く: ピアソン r={r_wo:.3f},  p={p_wo:.4f}')

# 参考：スピアマン（順位でみる相関。外れ値の影響を受けにくい・教科書外）
rs, ps = stats.spearmanr(m['女性正規率'], m['女流出率'])
print(f'全県(参考): スピアマン r={rs:.3f},  p={ps:.4f}')

結果（この授業の核心とつながります）：
- **全県** … r=0.236, **p ≈ 0.110**（＝**約9回に1回**は偶然でも起こる＝ありふれている）。p>0.05 なので「偶然では説明しにくい」とは**言い切れません**（有意でない）。
- **東京を除く** … r=0.492, **p ≈ 0.0005**（＝**約2000回に1回**＝めったに起きない）。今度は有意になります。

つまり、**「有意かどうか」さえ、東京という1点を入れるかどうかで逆転**します。ここから大事なこと：

- 検定（p値）が答えるのは **「偶然で説明できるか」だけ**。
  **「因果か」「外れ値に頑健か」「交絡はないか」には、何も答えません。**
- だから **p<0.05（有意）でも、それは「確証」ではありません**。探索（仮説生成）の段階のままです。
- 逆に、有意でなくても「無関係」と決まるわけではありません（今回、順位でみる相関＝スピアマンでは全県でも有意でした）。

> 📌 まとめ：検定は便利な道具ですが、**「有意＝正しい・因果がある」ではありません**。
> 散布図・外れ値・交絡・記述と因果の区別 ―― 今日のこれらの話の**置き換えにはなりません**。
> （検定の中身は来学期以降の統計の授業で。ここでは「p値が答えること／答えないこと」だけ持ち帰れば十分です。）


## 7. 記述として頑健なこと ― 男女で比べる ▶

因果は言えなくても、**記述**としてはっきり言えることもあります。
同じ流出率を男女で比べてみましょう。


In [ ]:
popM = (s[(s['T_Gender']=='男') & (s['T_Age']=='20～24歳')]
        .groupby('T_Prefecture')['Weight'].sum().rename('男20_24人口'))
mig_m = body[(body[1]=='60000') & (body[5].apply(is_pref))].copy()
mig_m = mig_m.rename(columns={6:'都道府県'})
mig_m['男20_24転入超過'] = pd.to_numeric(mig_m[35])  # 列35 = 男×20〜24歳

m = m.merge(mig_m[['都道府県','男20_24転入超過']], on='都道府県').merge(popM, left_on='都道府県', right_index=True)
m['男流出率'] = -m['男20_24転入超過'] / m['男20_24人口'] * 100
n = (m['女流出率'] > m['男流出率']).sum()
print(f'女性の流出率 > 男性の流出率 となる県: {n} / {len(m)}')
m.sort_values('女流出率', ascending=False)[['都道府県','女流出率','男流出率']].head(6).round(2)

47県中の多く（約8割＝37県）で、**若年女性の流出率が男性を上回ります**。
栃木のように、男性は転入超過なのに女性は転出超過、という県もあります。
これは「なぜ」には踏み込まない**記述**ですが、比較的頑健で、
「機会格差を**可視化**する」というコンテストの主目的には十分な強さの証拠です。

**記述（格差の地図を描く）と因果（なぜ起きるか）を分ける** ―― これが今日の核心です。


## 事前学習チェックリスト（授業前に確認）

授業に来る前に、次がすべて済んでいることを確認してください。

- [ ] §2〜§7 を上から順に**すべて実行**し、エラーが残っていない
- [ ] **全県の相関 r**（§5の出力）をメモした
- [ ] **東京を除いた r**（§6の出力）をメモした
- [ ] **女性の流出率が男性を上回る県数**（§7の出力）をメモした
- [ ] うまく動かなかった箇所・疑問点を1つ書き出した（授業の質問用）
- [ ] **確認テスト（Moodle）を締切までに提出**した

> 授業では、この結果を題材に **「結論か仮説か」「相関≠因果」「記述と因果の分離」** を解説します。
> 上の数値が手元にある状態で来てください。


## 8. （発展・応用）確証を試す ― 傾向スコア・マッチング

> 🧑‍🏫 **ここは授業中に一緒にやります**（事前学習は §7 までで大丈夫）。結果が有効でも無効でも、**方法の使い方と限界**を学ぶのが目的です。

§5〜§6で、「正規率が高い県ほど流出も多い」という向きが（弱く・東京で動きつつ）見えました。その背景に **都市度という交絡**（§6）を疑いました。
第8回でやった **傾向スコア・マッチング** は、まさにこの交絡を「**そろえて比べる**」ための道具です。実際に使って、確証に一歩近づけるか試してみましょう。

**やること**
- **処置**＝女性正規率が高い県（中央値以上） vs 低い県。
- **交絡候補**＝都市度（県の人口の対数）と女性有業率。
- 「高正規率県になりやすい度合い（＝傾向スコア）」が近い県どうしを対にして、**流出率の差**を「調整なし」と「マッチ後」で比べます。


In [ ]:
# まず交絡の候補を作る：都市度の代理＝県の15歳以上人口（対数）、もう一つ＝女性有業率
import numpy as np
県人口 = s.groupby('T_Prefecture')['Weight'].sum().rename('県人口')
s['有業W'] = (s['T_WorkStatus']=='有業者') * s['Weight']
女性有業率 = (s[s['T_Gender']=='女'].groupby('T_Prefecture')
            .apply(lambda x: x['有業W'].sum()/x['Weight'].sum()*100, include_groups=False)
            .rename('女性有業率'))
m = m.merge(県人口, left_on='都道府県', right_index=True).merge(女性有業率, left_on='都道府県', right_index=True)
m['log県人口'] = np.log(m['県人口'])
m[['都道府県','女性正規率','女流出率','県人口','女性有業率']].head()

In [ ]:
from sklearn.linear_model import LogisticRegression   # Colab に最初から入っています

# ① 処置：女性正規率が高い県（中央値以上）=1、低い県=0
med = m['女性正規率'].median()
m['処置'] = (m['女性正規率'] >= med).astype(int)

# ② 素朴な差（調整なし）：高正規率県 − 低正規率県 の流出率
naive = m.loc[m['処置']==1, '女流出率'].mean() - m.loc[m['処置']==0, '女流出率'].mean()

# ③ 傾向スコア：都市度(log県人口)と女性有業率から「高正規率県になりやすさ」を推定
X = m[['log県人口', '女性有業率']].values
m['傾向スコア'] = LogisticRegression().fit(X, m['処置']).predict_proba(X)[:, 1]

# ④ 1:1 最近傍マッチング：各「高正規率県」に、傾向スコアが最も近い「低正規率県」を当てる
t = m[m['処置']==1]; c = m[m['処置']==0]
cps, cout, clog = c['傾向スコア'].values, c['女流出率'].values, c['log県人口'].values
diffs, matched_log = [], []
for _, tr in t.iterrows():
    j = np.abs(cps - tr['傾向スコア']).argmin()    # 都市度などが似た対照県
    diffs.append(tr['女流出率'] - cout[j]); matched_log.append(clog[j])
att = np.mean(diffs)

print(f'処置(高正規率県) {len(t)}県 / 対照(低正規率県) {len(c)}県   （正規率の中央値 = {med:.1f}%）')
print(f'素朴な差（調整なし, 高−低の流出率）: {naive:+.2f}')
print(f'傾向スコアマッチ後の差（ATT近似）  : {att:+.2f}')
print(f'[バランス] log県人口の平均  処置:{t["log県人口"].mean():.2f} / 対照(調整前):{c["log県人口"].mean():.2f} / 対照(マッチ後):{np.mean(matched_log):.2f}')

### 結果の読み方 ― そして「これは確証か？」

- **素朴な差**（高正規率県 − 低正規率県の流出率）は **+2.3** くらい。探索で見えた「正規率が高い県ほど流出も多い(?)」という向きです。
- ところが **都市度（県人口）などをそろえてマッチング**すると、差は **+0.3** くらいまで縮みます（バランスも改善：対照のlog県人口が処置とほぼ一致）。
  → 見かけの関係の**多くは「都市度」という交絡で説明できそう**、と読めます。傾向スコアは「**似た都市度の県どうしを比べる**」道具だからです。

でも、いちばん大事なこと：**「やってみた」結果がどうであれ、これは因果の確証にはなりません。** 理由は——
- 47県という**集団（アグリゲート）**の比較で、個人の因果は語れない（**生態学的誤謬**）。
- そろえたのは都市度・有業率という**少数の代理だけ**。ほかの交絡（進学・産業構成…）は残る。
- **断面データ**で、移動の前後という**時間方向**がない。
- 就業構造側は**擬似データ**。

> 📌 まとめ：傾向スコア・マッチングは「**交絡をそろえて比べる**」確証寄りの道具で、今日はそれを**使ってみました**。
> 確証に近づくには **個人単位データ・より多くの交絡の調整・時間方向** が要ります。
> 「**探索で見えた → 交絡を疑う → そろえて比べる**」という*手順*こそ、持ち帰ってほしいものです。


## 9. まとめと演習

### 今日の要点
- 個票（自分で集計）と集計表（既製）を**県を鍵に結合**して探索できる。
- 相関は**弱く・予想と逆・外れ値で動いた**。強い相関でも因果ではないし、弱ければなおさら。
- **散布図と外れ値を必ず見る**。相関係数だけを信じない。
- 探索＝仮説生成、確証＝別の手続き。レポートでは**「関連は見られた。ただし因果は言えない」**と書く。
- 一方、男女差のような**記述**は頑健に言えることがある。記述と因果を分ける。

### データの適切な利用
就業構造側は擬似データ。実数の移動データと混ぜて出した相関は、授業用であり実証結論にしない。
出典明記（就業構造基本調査 一般用ミクロデータ／住民基本台帳人口移動報告）も忘れずに。

### 演習
1. 流出率の分母や対象年齢（15〜19、25〜29、または15〜29の合算）を変えると、相関や順位はどう変わるか。
2. 移動者の国籍を **日本人移動者（国籍コード `61000`）** に変えると（地方創生の分析でよく使う定義）、
   数字や相関はどう動くか。`body[1]=='60000'` を `'61000'` にして確かめよ。**定義が変われば数字も変わる**。
3. この分析で「言えること」と「言えないこと」を、各2つずつ箇条書きにせよ。
